# PDE vs ODE Comparison

Solve the full HJB PDE at nominal and extreme parameter values to verify
that the ODE approximation holds across the parameter ranges identified
in the sensitivity analysis.

For each of the 4 important parameters (sigma, gamma, lambda, eta),
we solve the PDE at the low and high end of the Sobol range and compare
QoIs against the ODE. 9 PDE solves total (1 nominal + 2 per parameter).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from src import build_paper_example_params, restrict_currencies, run_multicurrency_mm
from src.model import ModelParams, PairParams, TierParams, BP
from src.pde import solve_hjb_implicit
from src.hamiltonian import optimal_delta_logistic
from src.policy import Hprime_execution_cost

## Parameter setup

Same 8 parameters and ranges as the sensitivity analysis (see `sensitivity_analysis.ipynb`).

In [2]:
PARAM_NAMES = [
    r"$\sigma_{EUR}$", r"$\gamma$", r"$\lambda_{scale}$",
    r"$\alpha_1$", r"$\alpha_2$",
    r"$\beta_1$", r"$\beta_2$", r"$\eta$"
]

PARAM_LABELS = [
    "sigma_EUR", "gamma", "lambda_scale",
    "alpha_1", "alpha_2", "beta_1", "beta_2", "eta"
]

# Nominal values in human-readable units
# sigma: bps, gamma: 1/M$, lambda_scale: multiplier,
# alpha: dimensionless, beta: 1/bps (paper units), eta: bps
NOMINAL = np.array([80.0, 20.0, 1.0, -1.9, -0.3, 11.0, 3.5, 1e-5])

RANGES = np.array([
    [56.0, 104.0],       # sigma_EUR (bps), +-30%
    [10.0, 40.0],        # gamma, +-50%
    [0.5, 1.5],          # lambda_scale, +-50%
    [-2.5, -1.3],        # alpha_1, +-30%
    [-0.9, 0.3],         # alpha_2, +-100%
    [7.7, 14.3],         # beta_1 (1/bps), +-30%
    [2.45, 4.55],        # beta_2 (1/bps), +-30%
    [0.5e-5, 2.0e-5],    # eta (bps), +-50%
])

QOI_NAMES = [
    r"$\delta^*(y{=}0)$ [bps]",
    r"$\delta^*(y{=}10) - \delta^*(y{=}0)$ [bps]",
    r"$\xi^*(y{=}10)$ [M$/day]",
]

N_PARAMS = len(PARAM_NAMES)
N_QOIS = len(QOI_NAMES)

In [3]:
# Cache base parameters to avoid rebuilding every call
_base = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])
_base_pp = _base.pairs[("EUR", "USD")]


def build_modified_params(params):
    """Build 2-currency (USD, EUR) ModelParams from parameter vector.

    params[0]: sigma_EUR in bps
    params[1]: gamma (risk aversion)
    params[2]: lambda_scale (multiplier on all arrival rates)
    params[3]: alpha_1 (logistic shift, aggressive tier)
    params[4]: alpha_2 (logistic shift, passive tier)
    params[5]: beta_1 (logistic slope, aggressive, in 1/bps paper units)
    params[6]: beta_2 (logistic slope, passive, in 1/bps paper units)
    params[7]: eta (execution cost quadratic, in bps)
    """
    sigma_eur, gamma, lam_scale, a1, a2, b1, b2, eta = params

    tiers = [
        TierParams(alpha=a1, beta=b1 * 1e4),
        TierParams(alpha=a2, beta=b2 * 1e4),
    ]

    pp = PairParams(
        pair=("EUR", "USD"),
        sizes_musd=_base_pp.sizes_musd,
        lambdas_per_day=_base_pp.lambdas_per_day * lam_scale,
        tiers=tiers,
        psi=_base_pp.psi,
        eta=eta * BP,
    )

    return ModelParams(
        currencies=["USD", "EUR"],
        ref_ccy="USD",
        sigma={"USD": 0.0, "EUR": sigma_eur * BP},
        corr={},
        k=_base.k,
        mu={"USD": 0.0, "EUR": 0.0},
        gamma=gamma,
        kappa=np.zeros((2, 2)),
        T_days=_base.T_days,
        pairs={("EUR", "USD"): pp},
    )


def evaluate_qois(params):
    """Solve ODE and return 3 QoIs.

    Returns: [delta*(y=0) in bps, inventory skew in bps, hedge rate in M$/day]
    """
    mp = build_modified_params(params)
    res = run_multicurrency_mm(mp, n_steps=500)

    y_flat = np.zeros(2)
    y_long = np.array([0.0, 10.0])  # 10 M$ long EUR

    # QoI 1: neutral markup (tier 1, z=1 M$) in bps
    delta_flat = res.markup(0, "EUR", "USD", 1.0, y_flat)

    # QoI 2: inventory skew in bps
    delta_long = res.markup(0, "EUR", "USD", 1.0, y_long)
    skew = delta_long - delta_flat

    # QoI 3: hedge rate at y=10 (M$/day)
    xi = res.hedge_rate("EUR", "USD", y_long)

    return np.array([delta_flat / BP, skew / BP, xi])

In [4]:

# PDE grid: [-150, 150] M$, dy=1.0
Y_MAX = 150
DY = 1.0
y_grids_pde = [np.arange(-Y_MAX, Y_MAX + DY, DY) for _ in range(2)]
idx_origin = len(y_grids_pde[0]) // 2
PDE_N_STEPS = 20
PDE_PI_TOL = 1e-4

print(f"PDE grid: [{-Y_MAX}, {Y_MAX}] M$, dy={DY}, points/axis={len(y_grids_pde[0])}")
print(f"PDE time steps: {PDE_N_STEPS}, PI tol: {PDE_PI_TOL}")


def pde_extract_qois(theta_0, y_grids, mp):
    """Extract the 3 QoIs from PDE solution theta(0, y).

    Same QoIs as evaluate_qois but computed from the PDE grid directly:
      QoI 1: delta*(y=0)  — optimal markup at flat inventory
      QoI 2: delta*(y=10) - delta*(y=0)  — inventory skew
      QoI 3: xi*(y=10)  — hedge rate at 10 M$ EUR
    """
    idx0 = len(y_grids[0]) // 2
    dy = y_grids[0][1] - y_grids[0][0]

    key = ("EUR", "USD")
    pp = mp.pairs[key]
    tier = pp.tiers[0]
    z = 1.0
    shift = int(round(z / dy))

    # Quoting: EUR pays (i=1), USD sells (j=0) -> dvec = [-1, +1]
    # Shifted point: [y_USD - z, y_EUR + z]

    # QoI 1: delta*(y=0)
    # p = (theta(y) - theta(y + z*dvec)) / z
    p_flat = (theta_0[idx0, idx0] - theta_0[idx0 - shift, idx0 + shift]) / z
    delta_flat = optimal_delta_logistic(p_flat, tier.alpha, tier.beta)

    # QoI 2: delta*(y=[0,10]) - delta*(y=0)
    y_eur_idx = 10  # 10 M$ EUR = 10 grid points
    p_long = (theta_0[idx0, idx0 + y_eur_idx] -
              theta_0[idx0 - shift, idx0 + y_eur_idx + shift]) / z
    delta_long = optimal_delta_logistic(p_long, tier.alpha, tier.beta)
    skew = delta_long - delta_flat

    # QoI 3: xi*(y=[0,10])
    # Gradient via central differences
    grad_usd = (theta_0[idx0 + 1, idx0 + y_eur_idx] -
                theta_0[idx0 - 1, idx0 + y_eur_idx]) / (2 * dy)
    grad_eur = (theta_0[idx0, idx0 + y_eur_idx + 1] -
                theta_0[idx0, idx0 + y_eur_idx - 1]) / (2 * dy)

    # Hedge EUR buy, USD sell: base = grad_EUR - grad_USD
    base = grad_eur - grad_usd

    # Market impact (small but included for correctness)
    k_eur = mp.k.get("EUR", 0.0)
    k_usd = mp.k.get("USD", 0.0)
    impact = k_eur * 10.0 * (1.0 + grad_eur) - k_usd * 0.0 * (1.0 + grad_usd)

    xi = Hprime_execution_cost(base + impact, pp.psi, pp.eta)

    return np.array([delta_flat / BP, skew / BP, xi])

PDE grid: [-150, 150] M$, dy=1.0, points/axis=301
PDE time steps: 20, PI tol: 0.0001


In [ ]:
# Define parameter configurations: nominal + edges for the 4 important parameters
configs = {}
configs["nominal"] = NOMINAL.copy()

for idx, label, lo, hi in [
    (0, "sigma", RANGES[0, 0], RANGES[0, 1]),
    (1, "gamma", RANGES[1, 0], RANGES[1, 1]),
    (2, "lambda", RANGES[2, 0], RANGES[2, 1]),
    (7, "eta", RANGES[7, 0], RANGES[7, 1]),
]:
    p_lo = NOMINAL.copy(); p_lo[idx] = lo
    p_hi = NOMINAL.copy(); p_hi[idx] = hi
    configs[f"{label}_low"] = p_lo
    configs[f"{label}_high"] = p_hi

# Solve ODE and PDE for each configuration
results_ode = {}
results_pde = {}

for name, params in configs.items():
    print(f"\n{'='*60}")
    print(f"Config: {name}")
    print(f"{'='*60}")

    # ODE (fast)
    results_ode[name] = evaluate_qois(params)

    # PDE (slow)
    mp = build_modified_params(params)
    pde_result = solve_hjb_implicit(
        y_grids_pde, mp, n_steps=PDE_N_STEPS,
        pi_tol=PDE_PI_TOL, pi_max_iter=50,
    )
    results_pde[name] = pde_extract_qois(pde_result['theta_0'], y_grids_pde, mp)

    # Print comparison
    print(f"\n  {'QoI':>10s}  {'ODE':>12s}  {'PDE':>12s}  {'rel diff':>10s}")
    for q in range(N_QOIS):
        ode_val = results_ode[name][q]
        pde_val = results_pde[name][q]
        rd = abs(ode_val - pde_val) / (abs(pde_val) + 1e-30)
        print(f"  {'QoI '+str(q+1):>10s}  {ode_val:12.4f}  {pde_val:12.4f}  {rd:10.2e}")


Config: nominal


HJB PDE solve (implicit): 100%|██████████| 20/20 [08:44<00:00, 26.23s/step]



         QoI           ODE           PDE    rel diff
       QoI 1        0.1822        0.1826    2.07e-03
       QoI 2        0.1261        0.1300    2.99e-02
       QoI 3    -1878.6063    -2180.7083    1.39e-01

Config: sigma_low


HJB PDE solve (implicit): 100%|██████████| 20/20 [18:55<00:00, 56.76s/step]



         QoI           ODE           PDE    rel diff
       QoI 1        0.1807        0.1809    7.92e-04
       QoI 2        0.0819        0.0865    5.37e-02
       QoI 3        0.0000        0.0000    0.00e+00

Config: sigma_high


...

HJB PDE solve (implicit): 100%|██████████| 20/20 [02:17<00:00,  6.86s/step]



         QoI           ODE           PDE    rel diff
       QoI 1        0.1843        0.1847    2.31e-03
       QoI 2        0.1946        0.1904    2.21e-02
       QoI 3    -5763.3521    -5633.1427    2.31e-02

Config: lambda_low


HJB PDE solve (implicit): 100%|██████████| 20/20 [01:49<00:00,  5.46s/step]



         QoI           ODE           PDE    rel diff
       QoI 1        0.1843        0.1844    9.81e-04
       QoI 2        0.1946        0.1688    1.53e-01
       QoI 3    -5763.3525    -4505.1169    2.79e-01

Config: lambda_high


HJB PDE solve (implicit):  60%|██████    | 12/20 [06:59<09:40, 72.54s/step]

In [ ]:
# Compare sensitivities: (QoI at edge - QoI at nominal) for ODE vs PDE
param_pairs = [
    ("sigma", "sigma_low", "sigma_high"),
    ("gamma", "gamma_low", "gamma_high"),
    ("lambda", "lambda_low", "lambda_high"),
    ("eta", "eta_low", "eta_high"),
]

qoi_short = [r"$\delta^*(y{=}0)$", "skew", r"$\xi^*$"]

print("Sensitivity comparison: Delta(QoI) = QoI(edge) - QoI(nominal)")
print("=" * 85)

for label, lo_key, hi_key in param_pairs:
    print(f"\n--- {label} ---")
    print(f"  {'QoI':>15s}  {'ODE low':>10s}  {'PDE low':>10s}  {'ODE high':>10s}  {'PDE high':>10s}")
    for q in range(N_QOIS):
        ode_lo = results_ode[lo_key][q] - results_ode["nominal"][q]
        pde_lo = results_pde[lo_key][q] - results_pde["nominal"][q]
        ode_hi = results_ode[hi_key][q] - results_ode["nominal"][q]
        pde_hi = results_pde[hi_key][q] - results_pde["nominal"][q]
        print(f"  {qoi_short[q]:>15s}  {ode_lo:+10.4f}  {pde_lo:+10.4f}  {ode_hi:+10.4f}  {pde_hi:+10.4f}")

In [ ]:
# Visual comparison: ODE vs PDE sensitivities side by side
fig, axes = plt.subplots(1, N_QOIS, figsize=(6 * N_QOIS, 5))

param_labels_short = [r"$\sigma$", r"$\gamma$", r"$\lambda$", r"$\eta$"]
x = np.arange(len(param_pairs))
width = 0.35

for q in range(N_QOIS):
    ax = axes[q]

    # Compute sensitivity magnitude: (high - low) for ODE and PDE
    ode_range = []
    pde_range = []
    for label, lo_key, hi_key in param_pairs:
        ode_range.append(results_ode[hi_key][q] - results_ode[lo_key][q])
        pde_range.append(results_pde[hi_key][q] - results_pde[lo_key][q])

    ax.bar(x - width/2, ode_range, width, label='ODE', color='steelblue')
    ax.bar(x + width/2, pde_range, width, label='PDE', color='coral')
    ax.set_ylabel('QoI(high) - QoI(low)')
    ax.set_title(QOI_NAMES[q], fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(param_labels_short)
    ax.legend(fontsize=10)
    ax.axhline(0, color='k', linewidth=0.5)

plt.suptitle('Sensitivity: ODE vs PDE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()